In [ ]:
!pip install osmnx geopandas shapely
!pip install folium geopandas

In [2]:
import requests
import pandas as pd
import geopandas as gpd
import os
import time

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
DRIVE_FOLDER = '/content/drive/MyDrive/SmartSite/Step 1: Extract/Crawl_Results'
os.makedirs(DRIVE_FOLDER, exist_ok=True)

In [10]:
import requests
import time
import pandas as pd
import geopandas as gpd
import os

# =====================================================================
# 1. HÀM CHIA 4 PHẦN TỰ ĐỘNG CHO THÀNH PHỐ LỚN
# =====================================================================
def split_into_4_parts(min_lat, max_lat, min_lng, max_lng):
    mid_lat = (min_lat + max_lat) / 2
    mid_lng = (min_lng + max_lng) / 2
    return [
        {"min_lat": mid_lat, "max_lat": max_lat, "min_lng": min_lng, "max_lng": mid_lng}, # Tây Bắc
        {"min_lat": mid_lat, "max_lat": max_lat, "min_lng": mid_lng, "max_lng": max_lng}, # Đông Bắc
        {"min_lat": min_lat, "max_lat": mid_lat, "min_lng": min_lng, "max_lng": mid_lng}, # Tây Nam
        {"min_lat": min_lat, "max_lat": mid_lat, "min_lng": mid_lng, "max_lng": max_lng}  # Đông Nam
    ]

# ĐỊNH NGHĨA TỌA ĐỘ GIỚI HẠN
cities_bounds = {
    "DaNang": [{"min_lat": 15.9200, "max_lat": 16.1700, "min_lng": 108.0600, "max_lng": 108.3200}],
    "HaNoi": split_into_4_parts(20.5500, 21.4000, 105.2800, 106.0200),
    "HCM": split_into_4_parts(10.3500, 11.1600, 106.3500, 106.8500)
}

# =====================================================================
# 2. ENGINE GỌI TRỰC TIẾP OVERPASS API
# =====================================================================
def fetch_poi_direct(city_name, parts):
    print(f"\n🚀 Đang cào POI cho: {city_name} (Gồm {len(parts)} phần)")
    all_pois = []
    url = "http://overpass-api.de/api/interpreter"

    # Chia làm 2 nhóm query
    query_groups = {
        "nhom_dan_cu": lambda s,w,n,e: f"""
        [out:json][timeout:90];
        (
          nwr["amenity"~"school|university|college|kindergarten"]({s},{w},{n},{e});
          nwr["building"~"office|apartments|residential|commercial"]({s},{w},{n},{e});
          nwr["office"]({s},{w},{n},{e});
          nwr["landuse"~"residential|commercial"]({s},{w},{n},{e});
        );
        out center;
        """,
                "nhom_fb_giaothong": lambda s,w,n,e: f"""
        [out:json][timeout:90];
        (
          nwr["amenity"~"restaurant|cafe|fast_food|bar|food_court"]({s},{w},{n},{e});
          nwr["amenity"~"bus_station|cinema|gym|library|marketplace"]({s},{w},{n},{e});
          nwr["highway"~"bus_stop"]({s},{w},{n},{e});
          nwr["railway"~"station|subway_entrance"]({s},{w},{n},{e});
          nwr["shop"~"supermarket|mall|convenience"]({s},{w},{n},{e});
          nwr["leisure"~"park"]({s},{w},{n},{e});
        );
        out center;
        """
    }

    for i, bounds in enumerate(parts):
        s, w, n, e = bounds["min_lat"], bounds["min_lng"], bounds["max_lat"], bounds["max_lng"]
        print(f" ⏳ Đang truy vấn Part {i+1}...")

        for group_name, query_fn in query_groups.items():
            query = query_fn(s, w, n, e)
            try:
                response = requests.post(url, data={'data': query})
                if response.status_code != 200:
                    wait = 30 if response.status_code == 429 else 20
                    print(f"   ⚠️ {group_name} bị ngộp (Code {response.status_code}). Đợi {wait}s...")
                    time.sleep(wait)
                    response = requests.post(url, data={'data': query})

                data = response.json()
                count = 0
                for element in data.get('elements', []):
                    lat = element.get('lat') or element.get('center', {}).get('lat')
                    lon = element.get('lon') or element.get('center', {}).get('lon')
                    if not lat or not lon:
                        continue

                    tags = element.get('tags', {})
                    name = tags.get('name', 'N/A')
                    amenity = tags.get('amenity', '')
                    building = tags.get('building', '')
                    landuse = tags.get('landuse', '')
                    shop = tags.get('shop', '')
                    railway = tags.get('railway', '')
                    leisure = tags.get('leisure', '')
                    highway = tags.get('highway', '')

                    poi_type = "Khác"
                    if amenity: poi_type = amenity
                    elif shop: poi_type = f"shop_{shop}"
                    elif railway: poi_type = f"railway_{railway}"
                    elif leisure: poi_type = f"leisure_{leisure}"
                    elif highway: poi_type = f"highway_{highway}"
                    elif building: poi_type = f"building_{building}"
                    elif landuse: poi_type = f"landuse_{landuse}"

                    all_pois.append({'name': name, 'type': poi_type, 'lat': lat, 'lon': lon})
                    count += 1

                print(f"   ✅ Part {i+1} [{group_name}]: {count} POI")
                time.sleep(10)

            except Exception as err:
                print(f"   ❌ Lỗi Part {i+1} [{group_name}]: {err}")

    if not all_pois:
        print(f"⚠️ Không lấy được dữ liệu cho {city_name}")
        return

    df = pd.DataFrame(all_pois)
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")
    filepath = os.path.join(DRIVE_FOLDER, f"{city_name}_POI_Points.geojson")
    gdf.to_file(filepath, driver="GeoJSON")
    print(f"🎉 HOÀN TẤT {city_name}! Tổng cộng {len(gdf)} POI → {filepath}")

In [11]:
fetch_poi_direct("DaNang", cities_bounds["DaNang"])


🚀 Đang cào POI cho: DaNang (Gồm 1 phần)
 ⏳ Đang truy vấn Part 1...
   ⚠️ nhom_dan_cu bị ngộp (Code 504). Đợi 20s...
   ✅ Part 1 [nhom_dan_cu]: 5749 POI
   ⚠️ nhom_fb_giaothong bị ngộp (Code 504). Đợi 20s...
   ✅ Part 1 [nhom_fb_giaothong]: 3166 POI
🎉 HOÀN TẤT DaNang! Tổng cộng 8915 POI → /content/drive/MyDrive/SmartSite/Step 1: Extract/Crawl_Results/DaNang_POI_Points.geojson


In [12]:
fetch_poi_direct("HaNoi", cities_bounds["HaNoi"])


🚀 Đang cào POI cho: HaNoi (Gồm 4 phần)
 ⏳ Đang truy vấn Part 1...
   ✅ Part 1 [nhom_dan_cu]: 820 POI
   ✅ Part 1 [nhom_fb_giaothong]: 1049 POI
 ⏳ Đang truy vấn Part 2...
   ⚠️ nhom_dan_cu bị ngộp (Code 504). Đợi 20s...
   ✅ Part 2 [nhom_dan_cu]: 7550 POI
   ✅ Part 2 [nhom_fb_giaothong]: 8761 POI
 ⏳ Đang truy vấn Part 3...
   ⚠️ nhom_dan_cu bị ngộp (Code 504). Đợi 20s...
   ✅ Part 3 [nhom_dan_cu]: 634 POI
   ✅ Part 3 [nhom_fb_giaothong]: 264 POI
 ⏳ Đang truy vấn Part 4...
   ⚠️ nhom_dan_cu bị ngộp (Code 429). Đợi 30s...
   ✅ Part 4 [nhom_dan_cu]: 1610 POI
   ⚠️ nhom_fb_giaothong bị ngộp (Code 504). Đợi 20s...
   ✅ Part 4 [nhom_fb_giaothong]: 1929 POI
🎉 HOÀN TẤT HaNoi! Tổng cộng 22617 POI → /content/drive/MyDrive/SmartSite/Step 1: Extract/Crawl_Results/HaNoi_POI_Points.geojson


In [18]:
fetch_poi_direct("HCM", cities_bounds["HCM"])


🚀 Đang cào POI cho: HCM (Gồm 4 phần)
 ⏳ Đang truy vấn Part 1...
   ✅ Part 1 [nhom_dan_cu]: 259 POI
   ⚠️ nhom_fb_giaothong bị ngộp (Code 504). Đợi 20s...
   ✅ Part 1 [nhom_fb_giaothong]: 1062 POI
 ⏳ Đang truy vấn Part 2...
   ⚠️ nhom_dan_cu bị ngộp (Code 429). Đợi 30s...
   ✅ Part 2 [nhom_dan_cu]: 4662 POI
   ✅ Part 2 [nhom_fb_giaothong]: 9909 POI
 ⏳ Đang truy vấn Part 3...
   ⚠️ nhom_dan_cu bị ngộp (Code 504). Đợi 20s...
   ✅ Part 3 [nhom_dan_cu]: 834 POI
   ✅ Part 3 [nhom_fb_giaothong]: 601 POI
 ⏳ Đang truy vấn Part 4...
   ✅ Part 4 [nhom_dan_cu]: 1874 POI
   ✅ Part 4 [nhom_fb_giaothong]: 2250 POI
🎉 HOÀN TẤT HCM! Tổng cộng 21451 POI → /content/drive/MyDrive/SmartSite/Step 1: Extract/Crawl_Results/HCM_POI_Points.geojson
